# Hoop Vision: End-to-End Pipeline Demo

This notebook demonstrates the complete Hoop Vision pipeline for predicting next possession outcomes from NBA video clips.

## Pipeline Overview

The pipeline consists of 5 stages:

1. **Video Ingestion**: Extract frames from video clips
2. **Event Detection**: Detect players, ball, and events using YOLO
3. **Graph Construction**: Build temporal event graphs
4. **Feature Extraction**: Extract engineered features from graphs
5. **Prediction/Training**: Train and evaluate prediction models

Each stage processes the output of the previous stage, creating a sequential pipeline from raw video to predictions.

## Setup: Imports and Configuration

In [ ]:
# Standard library imports
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Data science imports
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns

# Hoop Vision pipeline imports
from hoop_vision.config import Config
from hoop_vision.stage1_ingestion import VideoIngestion
from hoop_vision.stage2_detection import EventDetection
from hoop_vision.stage3_graph import GraphBuilder
from hoop_vision.stage4_features import FeatureExtractor
from hoop_vision.stage5_prediction import PredictionModel

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All imports successful")

In [ ]:
# Load pipeline configuration
config = Config('config.yaml')

print("Pipeline Configuration:")
print(f"  Ingestion FPS: {config.fps}")
print(f"  YOLO Model: {config.yolo_model}")
print(f"  Confidence Threshold: {config.confidence_threshold}")
print(f"  Max Temporal Distance: {config.max_temporal_distance}s")
print(f"  Lookback Window: {config.lookback_window} possessions")
print(f"  Model Type: {config.model_type}")
print(f"  Test Split: {config.test_split}")
print(f"\nData Directories:")
print(f"  Raw Clips: {config.raw_clips_dir}")
print(f"  Frames Output: {config.frames_output_dir}")
print(f"  Events Output: {config.events_output_dir}")
print(f"  Graphs Output: {config.graphs_output_dir}")
print(f"  Features Output: {config.features_output_dir}")
print(f"  Predictions Output: {config.predictions_output_dir}")

## Stage 1: Video Ingestion

Extract frames from video clips at a target FPS (frames per second).

**Input**: Raw video clips (.mp4, .mov)

**Output**: Extracted frames (JPEG images) + metadata JSON

**Purpose**: Convert video into discrete frames for analysis, reducing data volume while preserving temporal information.

In [ ]:
# Initialize ingestion stage
ingestion = VideoIngestion(config)

# Check for available clips
clips_dir = config.raw_clips_dir
video_files = list(clips_dir.glob('*.mp4')) + list(clips_dir.glob('*.mov'))

print(f"Found {len(video_files)} video clips in {clips_dir}")
for video in video_files[:5]:  # Show first 5
    print(f"  - {video.name}")
if len(video_files) > 5:
    print(f"  ... and {len(video_files) - 5} more")

In [ ]:
# Process all clips (extract frames)
# WARNING: This may take several minutes depending on number and length of clips

print("Processing video clips...")
all_metadata = ingestion.process_all_clips(clips_dir)

print(f"\n=== Stage 1 Summary ===")
print(f"Processed {len(all_metadata)} clips")
print(f"Total frames extracted: {sum(m['num_frames'] for m in all_metadata)}")
print(f"Output directory: {config.frames_output_dir}")

In [ ]:
# Examine metadata from first clip
if all_metadata:
    sample_metadata = all_metadata[0]
    print("Sample clip metadata:")
    print(json.dumps(sample_metadata, indent=2))
    
    # Visualize frame extraction statistics
    frame_counts = [m['num_frames'] for m in all_metadata]
    durations = [m['source_duration'] for m in all_metadata]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].hist(frame_counts, bins=20, edgecolor='black')
    axes[0].set_xlabel('Number of Frames')
    axes[0].set_ylabel('Number of Clips')
    axes[0].set_title('Distribution of Extracted Frames per Clip')
    
    axes[1].hist(durations, bins=20, edgecolor='black', color='green')
    axes[1].set_xlabel('Duration (seconds)')
    axes[1].set_ylabel('Number of Clips')
    axes[1].set_title('Distribution of Clip Durations')
    
    plt.tight_layout()
    plt.show()

## Stage 2: Event Detection

Detect objects (players, ball, etc.) in each frame using YOLOv8.

**Input**: Extracted frames from Stage 1

**Output**: Event detections JSON (bounding boxes, classes, confidence scores)

**Purpose**: Identify relevant basketball events and entities in each frame for downstream analysis.

In [ ]:
# Initialize event detection stage
detection = EventDetection(config)

print(f"Loaded YOLO model: {config.yolo_model}")
print(f"Confidence threshold: {config.confidence_threshold}")
print(f"NMS threshold: {config.nms_threshold}")

In [ ]:
# Process all clips (detect events in frames)
# WARNING: This may take several minutes, especially without GPU acceleration

print("Detecting events in frames...")
detection.process_all_clips()

print(f"\n=== Stage 2 Summary ===")
print(f"Event detection complete")
print(f"Output directory: {config.events_output_dir}")

In [ ]:
# Examine events from first clip
events_dirs = list(config.events_output_dir.glob('*'))
if events_dirs:
    sample_events_file = events_dirs[0] / 'events.json'
    with open(sample_events_file) as f:
        events = json.load(f)
    
    print(f"Sample events from clip: {events_dirs[0].name}")
    print(f"Total events: {len(events)}")
    print(f"\nFirst event:")
    print(json.dumps(events[0], indent=2))
    
    # Count detections per frame
    detection_counts = [len(event['detections']) for event in events]
    
    # Collect all detected classes
    all_classes = []
    for event in events:
        for detection in event['detections']:
            all_classes.append(detection['class_name'])
    
    # Visualize detection statistics
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(detection_counts)
    axes[0].set_xlabel('Frame Index')
    axes[0].set_ylabel('Number of Detections')
    axes[0].set_title('Detections per Frame')
    axes[0].grid(True, alpha=0.3)
    
    class_counts = pd.Series(all_classes).value_counts()
    axes[1].barh(class_counts.index, class_counts.values)
    axes[1].set_xlabel('Count')
    axes[1].set_ylabel('Class')
    axes[1].set_title('Detected Object Classes')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nAverage detections per frame: {np.mean(detection_counts):.2f}")
    print(f"Max detections in a frame: {max(detection_counts)}")
    print(f"Total unique classes detected: {len(class_counts)}")

## Stage 3: Graph Construction

Build temporal directed graphs connecting events across frames.

**Input**: Event detections from Stage 2

**Output**: NetworkX graph (nodes = frame events, edges = temporal connections)

**Purpose**: Capture temporal relationships between events for graph-based feature extraction and learning.

In [ ]:
# Initialize graph construction stage
graph_builder = GraphBuilder(config)

print(f"Max temporal distance: {config.max_temporal_distance}s")
print("Building temporal graphs...")

In [ ]:
# Process all clips (build graphs)
graph_builder.process_all_clips()

print(f"\n=== Stage 3 Summary ===")
print(f"Graph construction complete")
print(f"Output directory: {config.graphs_output_dir}")

In [ ]:
# Examine graph from first clip
graph_dirs = list(config.graphs_output_dir.glob('*'))
if graph_dirs:
    sample_graph_file = graph_dirs[0] / 'graph.gpickle'
    G = nx.read_gpickle(sample_graph_file)
    
    print(f"Sample graph from clip: {graph_dirs[0].name}")
    print(f"Nodes: {G.number_of_nodes()}")
    print(f"Edges: {G.number_of_edges()}")
    print(f"Density: {nx.density(G):.4f}")
    print(f"\nSample node attributes:")
    sample_node = list(G.nodes())[0]
    print(json.dumps(G.nodes[sample_node], indent=2, default=str))

### Visualize Graph Structure

In [ ]:
# Visualize the temporal graph structure
if graph_dirs:
    # Calculate degree distribution
    in_degrees = [G.in_degree(node) for node in G.nodes()]
    out_degrees = [G.out_degree(node) for node in G.nodes()]
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # In-degree distribution
    axes[0, 0].hist(in_degrees, bins=20, edgecolor='black', color='blue', alpha=0.7)
    axes[0, 0].set_xlabel('In-Degree')
    axes[0, 0].set_ylabel('Number of Nodes')
    axes[0, 0].set_title('In-Degree Distribution')
    
    # Out-degree distribution
    axes[0, 1].hist(out_degrees, bins=20, edgecolor='black', color='green', alpha=0.7)
    axes[0, 1].set_xlabel('Out-Degree')
    axes[0, 1].set_ylabel('Number of Nodes')
    axes[0, 1].set_title('Out-Degree Distribution')
    
    # Visualize small subgraph (first 15 nodes)
    subgraph = G.subgraph(list(G.nodes())[:15])
    pos = nx.spring_layout(subgraph, seed=42)
    
    # Draw subgraph
    nx.draw(subgraph, pos, ax=axes[1, 0],
            node_color='lightblue',
            node_size=500,
            with_labels=True,
            font_size=8,
            arrows=True,
            arrowsize=10,
            edge_color='gray',
            width=1.5)
    axes[1, 0].set_title('Temporal Graph Subgraph (First 15 Nodes)')
    axes[1, 0].axis('off')
    
    # Node detection count over time
    node_data = [(G.nodes[node]['frame_idx'], G.nodes[node]['num_detections']) 
                 for node in G.nodes()]
    node_data.sort()
    frame_indices, detection_counts = zip(*node_data)
    
    axes[1, 1].scatter(frame_indices, detection_counts, alpha=0.5)
    axes[1, 1].set_xlabel('Frame Index')
    axes[1, 1].set_ylabel('Number of Detections')
    axes[1, 1].set_title('Detections per Node (Temporal Order)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nGraph Statistics:")
    print(f"Average in-degree: {np.mean(in_degrees):.2f}")
    print(f"Average out-degree: {np.mean(out_degrees):.2f}")
    print(f"Max in-degree: {max(in_degrees)}")
    print(f"Max out-degree: {max(out_degrees)}")

## Stage 4: Feature Extraction

Extract engineered features from temporal graphs.

**Input**: Temporal graphs from Stage 3

**Output**: Feature vectors (CSV) for each node/frame

**Purpose**: Transform graph structure and detection data into numerical features for machine learning.

In [ ]:
# Initialize feature extraction stage
feature_extractor = FeatureExtractor(config)

print(f"Lookback window: {config.lookback_window} possessions")
print("Extracting features...")

In [ ]:
# Process all clips (extract features)
feature_extractor.process_all_clips()

print(f"\n=== Stage 4 Summary ===")
print(f"Feature extraction complete")
print(f"Output directory: {config.features_output_dir}")

In [ ]:
# Examine features from first clip
features_dirs = list(config.features_output_dir.glob('*'))
if features_dirs:
    sample_features_file = features_dirs[0] / 'features.csv'
    features_df = pd.read_csv(sample_features_file)
    
    print(f"Sample features from clip: {features_dirs[0].name}")
    print(f"Number of samples: {len(features_df)}")
    print(f"Number of features: {len(features_df.columns)}")
    print(f"\nFeature columns:")
    print(features_df.columns.tolist())
    print(f"\nFirst few rows:")
    display(features_df.head())
    
    print(f"\nFeature statistics:")
    display(features_df.describe())

In [ ]:
# Visualize feature distributions
if features_dirs:
    # Select numeric columns only
    numeric_cols = features_df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Plot distributions of key features
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Feature Distributions', fontsize=16)
    
    key_features = ['num_detections', 'in_degree', 'out_degree', 'lookback_avg_detections']
    available_features = [f for f in key_features if f in numeric_cols]
    
    for idx, feature in enumerate(available_features[:4]):
        ax = axes[idx // 2, idx % 2]
        ax.hist(features_df[feature].dropna(), bins=20, edgecolor='black', alpha=0.7)
        ax.set_xlabel(feature)
        ax.set_ylabel('Frequency')
        ax.set_title(f'Distribution of {feature}')
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # Correlation heatmap
    if len(numeric_cols) > 1:
        plt.figure(figsize=(12, 10))
        correlation_matrix = features_df[numeric_cols].corr()
        sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                    center=0, square=True, linewidths=1)
        plt.title('Feature Correlation Matrix')
        plt.tight_layout()
        plt.show()

## Stage 5: Prediction/Training

Train and evaluate machine learning models for next possession prediction.

**Input**: Feature vectors from Stage 4 + ground truth labels

**Output**: Trained model + evaluation metrics

**Purpose**: Learn to predict next possession outcomes (made shot, miss, turnover, foul, etc.)

**Note**: This stage requires labeled data. If no labels are available, dummy labels will be generated for demonstration purposes.

In [ ]:
# Initialize prediction model
prediction_model = PredictionModel(config)

print(f"Model type: {config.model_type}")
print(f"Test split: {config.test_split}")
print(f"Random seed: {config.random_seed}")

In [ ]:
# Load all features from processed clips
all_features = prediction_model.load_all_features()

print(f"Loaded features:")
print(f"  Total samples: {len(all_features)}")
print(f"  Number of clips: {all_features['clip_id'].nunique()}")
print(f"  Feature columns: {len(all_features.columns)}")
print(f"\nFeature preview:")
display(all_features.head())

In [ ]:
# Train baseline model
# Note: This will use dummy labels if no real labels file exists

print("Training baseline XGBoost model...")
results = prediction_model.train_baseline()

print(f"\n=== Stage 5 Summary ===")
print(f"Model saved to: {results['model_path']}")
print(f"Baseline accuracy: {results['metrics']['accuracy']:.3f}")

In [ ]:
# Visualize model performance
metrics = results['metrics']

print(f"\nModel Training Summary:")
print(f"  Train samples: {metrics['num_train_samples']}")
print(f"  Test samples: {metrics['num_test_samples']}")
print(f"  Number of features: {metrics['num_features']}")
print(f"  Accuracy: {metrics['accuracy']:.3f}")

# Extract per-class metrics
report = metrics['classification_report']
classes = [k for k in report.keys() if k not in ['accuracy', 'macro avg', 'weighted avg']]

if classes:
    precisions = [report[c]['precision'] for c in classes]
    recalls = [report[c]['recall'] for c in classes]
    f1_scores = [report[c]['f1-score'] for c in classes]
    
    # Plot per-class metrics
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    x = np.arange(len(classes))
    width = 0.6
    
    axes[0].bar(x, precisions, width, color='blue', alpha=0.7)
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Precision')
    axes[0].set_title('Precision by Class')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(classes, rotation=45, ha='right')
    axes[0].set_ylim([0, 1.0])
    axes[0].grid(True, alpha=0.3)
    
    axes[1].bar(x, recalls, width, color='green', alpha=0.7)
    axes[1].set_xlabel('Class')
    axes[1].set_ylabel('Recall')
    axes[1].set_title('Recall by Class')
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(classes, rotation=45, ha='right')
    axes[1].set_ylim([0, 1.0])
    axes[1].grid(True, alpha=0.3)
    
    axes[2].bar(x, f1_scores, width, color='orange', alpha=0.7)
    axes[2].set_xlabel('Class')
    axes[2].set_ylabel('F1-Score')
    axes[2].set_title('F1-Score by Class')
    axes[2].set_xticks(x)
    axes[2].set_xticklabels(classes, rotation=45, ha='right')
    axes[2].set_ylim([0, 1.0])
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Pipeline Summary

Congratulations! You've successfully run the complete Hoop Vision pipeline:

1. **Stage 1 (Ingestion)**: Extracted frames from video clips
2. **Stage 2 (Detection)**: Detected objects and events using YOLO
3. **Stage 3 (Graph)**: Built temporal event graphs
4. **Stage 4 (Features)**: Extracted engineered features
5. **Stage 5 (Prediction)**: Trained baseline prediction model

### Key Outputs

- **Frames**: `data/processed/frames/`
- **Events**: `data/processed/events/`
- **Graphs**: `data/processed/graphs/`
- **Features**: `data/processed/features/`
- **Predictions**: `data/processed/predictions/`
- **Trained Model**: `models/checkpoints/xgboost_baseline.json`

## Next Steps

Now that you have a working baseline pipeline, here are some ideas for improvement:

### 1. Data Collection
- Collect more NBA video clips for training
- Create ground truth labels for possession outcomes
- Ensure diverse game situations (different teams, venues, game states)

### 2. Feature Engineering
- Add player tracking features (positions, velocities)
- Extract shot attempt features (release angle, distance)
- Include game context (score differential, time remaining)
- Add team-level features (offensive/defensive ratings)

### 3. Model Improvements
- Experiment with graph neural networks (GNNs)
- Try temporal models (LSTMs, Transformers)
- Implement ensemble methods
- Tune hyperparameters using cross-validation

### 4. Evaluation
- Implement proper cross-validation (by game/season)
- Add evaluation metrics (log loss, calibration)
- Create confusion matrix visualization
- Analyze errors by possession type

### 5. Detection Improvements
- Fine-tune YOLO on basketball-specific data
- Add custom object classes (ball, hoop, court lines)
- Implement player tracking/re-identification
- Extract pose estimation for shot mechanics

### 6. Graph Enhancements
- Add spatial edges (player-player interactions)
- Include possession-level subgraphs
- Weight edges by feature similarity
- Experiment with different temporal windows

### 7. Deployment
- Create real-time prediction API
- Build web interface for predictions
- Implement batch processing pipeline
- Add monitoring and logging

### 8. Documentation
- Document data collection process
- Create annotation guidelines
- Write model card for deployed models
- Publish performance benchmarks

## Resources

- **Project Repository**: [Link to GitHub repo]
- **Documentation**: See `docs/` directory
- **YOLOv8 Docs**: https://docs.ultralytics.com/
- **NetworkX Docs**: https://networkx.org/documentation/stable/
- **XGBoost Docs**: https://xgboost.readthedocs.io/

For questions or contributions, please open an issue or pull request on GitHub.